In [0]:
-- Clientes mais antigos, tem mais frequência de transação?
-- SELECT idCliente,
--        DtCriacao,
--        datediff('2026-02-04', DtCriacao) AS idadeBase

-- FROM workspace.tmw_points.clientes
-- WHERE YEAR(DtCriacao) = 2025
-- AND MONTH(DtCriacao) = 1

-- TRANSAÇÕES DOS ÚLTIMOS 365 DIAS

SELECT t1.*,
       t2.*

FROM (

    SELECT IdCliente,
          count(IdTransacao) AS qtdeTransacoes,
          min(DtCriacao)

    FROM workspace.tmw_points.transacoes
    WHERE datediff('2026-02-04', DtCriacao)  <= 365
    GROUP BY IdCliente

) AS t1

INNER JOIN (
    SELECT idCliente,
          DtCriacao,
          datediff('2026-02-04', DtCriacao) AS idadeBase

    FROM workspace.tmw_points.clientes
    WHERE YEAR(DtCriacao) = 2025
    AND MONTH(DtCriacao) = 1
) as T2
ON t1.idcliente = t2.idcliente

In [0]:
WITH tb_safra AS (

    SELECT IdCliente,
          count(IdTransacao) AS qtdeTransacoes,
          min(DtCriacao)

    FROM workspace.tmw_points.transacoes
    WHERE datediff('2026-02-04', DtCriacao)  <= 365
    GROUP BY IdCliente
),

tb_transacoes AS (

    SELECT idCliente,
          DtCriacao,
          datediff('2026-02-04', DtCriacao) AS idadeBase

    FROM workspace.tmw_points.clientes
    WHERE (YEAR(DtCriacao) = 2025
    AND MONTH(DtCriacao) = 1)
    OR (YEAR(DtCriacao) = 2024
    AND MONTH(DtCriacao) in (10,11,12))

)

SELECT *
FROM tb_transacoes AS t1
INNER JOIN tb_safra AS t2
ON t1.idcliente = t2.idCliente


Databricks visualization. Run in Databricks to view.

In [0]:
-- Quantidade de transações Acumuladas ao longo do tempo do sistema? Visão diária.
WITH tb_dia AS (
    SELECT date(DtCriacao) AS dtDia,
          count(IdTransacao) AS qtdeTransacoes
    FROM workspace.tmw_points.transacoes
    GROUP BY dtDia
)

SELECT t1.dtdia,
       t1.qtdeTransacoes,
       sum(t2.qtdeTransacoes) AS sumAcum,
       count(*)

FROM tb_dia AS t1
LEFT JOIN tb_dia AS t2
ON t1.dtDia >= t2.dtdia

GROUP BY ALL
ORDER BY t1.dtDia

Databricks visualization. Run in Databricks to view.

In [0]:
-- Quantidade de transações Acumuladas ao longo do tempo do sistema? Visão diária.
WITH tb_dia AS (
    SELECT date(DtCriacao) AS dtDia,
          count(IdTransacao) AS qtdeTransacoes
    FROM workspace.tmw_points.transacoes
    GROUP BY dtDia
),

tb_window AS (
  SELECT t1.dtdia,
        t1.qtdeTransacoes,
        sum(t1.qtdeTransacoes) OVER (ORDER BY t1.dtdia) AS totalAcum,
        sum(t1.qtdeTransacoes) OVER (ORDER BY t1.qtdeTransacoes) AS totalAcumQtde,
        t1.qtdeTransacoes - lag(t1.qtdeTransacoes) OVER (ORDER BY t1.dtdia) AS lagTransacao

  FROM tb_dia AS t1
)

SELECT *
FROM tb_window

In [0]:
-- Saldo de pontos acumulado de cada usuário
WITH tb_cliente_dia AS (
    SELECT IdCliente,
          date(DtCriacao) as dtDia,
          sum(QtdePontos) AS qtdePontos
    FROM workspace.tmw_points.transacoes
    GROUP BY ALL
)

SELECT *,
       sum(qtdePontos) OVER (PARTITION BY IdCliente ORDER BY dtDia) AS pontosAcum
FROM tb_cliente_dia

In [0]:
WITH tb_cliente_dia AS (
    SELECT DISTINCT IdCliente,
          date(DtCriacao)
    FROM workspace.tmw_points.transacoes
),

tb_lag AS (
    SELECT *,
          LAG(DtCriacao) OVER (PARTITION BY idcliente ORDER BY DtCriacao) AS lagDia
    FROM tb_cliente_dia
),

tb_cliente_diff AS (

    SELECT idcliente,
          avg(date_diff(DtCriacao, lagDia)) AS diffDates

    FROM tb_lag
    GROUP BY idcliente
    HAVING diffDates IS NOT NULL
    -- HAVING COUNT(*) > 1

),

tb_diff_count AS (

    SELECT diffDates,
          COUNT(*) as qtdeClientes

    FROM tb_cliente_diff
    GROUP BY diffDates
    ORDER BY diffDates
)

SELECT *,
      SUM(qtdeClientes) OVER (order by diffDates) AS qtdeClientesAcum,
      SUM(qtdeClientes) OVER (order by diffDates) / SUM(qtdeClientes) OVER () AS propClientAcum

FROM tb_diff_count

Databricks visualization. Run in Databricks to view.